# Notebook 1 — IndiaFinBench: Data Exploration

This notebook explores the corpus and QA dataset used in IndiaFinBench.
All paths are relative to the repository root — run from `IndiaFinBench/`.

In [ ]:
import json
import re
import warnings
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

# Paths
ROOT          = Path('..')
META_SEBI     = ROOT / 'data/metadata_sebi.csv'
META_RBI      = ROOT / 'data/metadata_rbi.csv'
PARSE_REPORT  = ROOT / 'data/parse_report.csv'
QA_JSON       = ROOT / 'annotation/raw_qa/indiafinbench_qa_combined_150.json'

print('Paths configured. All files exist:')
for p in [META_SEBI, META_RBI, PARSE_REPORT, QA_JSON]:
    print(f'  {p.name}: {p.exists()}')

In [ ]:
# Cell 2 — Load metadata CSVs, print shape and column dtypes
sebi_meta = pd.read_csv(META_SEBI)
rbi_meta  = pd.read_csv(META_RBI)
meta      = pd.concat([sebi_meta, rbi_meta], ignore_index=True)

print(f'SEBI metadata: {sebi_meta.shape[0]} rows x {sebi_meta.shape[1]} cols')
print(f'RBI  metadata: {rbi_meta.shape[0]}  rows x {rbi_meta.shape[1]} cols')
print(f'Combined:      {meta.shape[0]} rows\n')
print('Column dtypes (combined):')
print(meta.dtypes.to_string())

In [ ]:
# Cell 3 — Document count by source (SEBI vs RBI)
source_counts = meta['source'].value_counts()
print('Document count by source:')
print(source_counts.to_string())

fig, ax = plt.subplots(figsize=(5, 3))
colors = ['#1f77b4', '#ff7f0e']
bars = ax.bar(source_counts.index, source_counts.values, color=colors, edgecolor='white')
for bar, val in zip(bars, source_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            str(val), ha='center', va='bottom', fontweight='bold')
ax.set_title('Document Count by Regulatory Source', fontweight='bold')
ax.set_ylabel('Number of Documents')
ax.set_ylim(0, source_counts.max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 4 — Category distribution pie chart
# Use 'category' column if present, otherwise derive from filename prefix
if 'category' in meta.columns and meta['category'].notna().sum() > 0:
    cat_col = meta['category'].fillna('uncategorised')
else:
    # Derive from filename patterns common in SEBI/RBI docs
    def infer_category(fn):
        fn = str(fn).lower()
        if 'circular' in fn:    return 'Circular'
        if 'master' in fn:      return 'Master Circular'
        if 'regulation' in fn:  return 'Regulation'
        if 'order' in fn:       return 'Order'
        if 'guideline' in fn:   return 'Guideline'
        if 'notification' in fn: return 'Notification'
        return 'Other'
    cat_col = meta['filename'].apply(infer_category)

cat_counts = cat_col.value_counts()
print('Category distribution:')
print(cat_counts.to_string())

fig, ax = plt.subplots(figsize=(6, 4))
wedge_props = {'edgecolor': 'white', 'linewidth': 1.5}
ax.pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
       wedgeprops=wedge_props, startangle=140)
ax.set_title('Document Category Distribution', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5 — Word count distribution histogram (from parse_report.csv)
parse_df = pd.read_csv(PARSE_REPORT)
print(f'Parse report: {parse_df.shape[0]} rows')
print(parse_df[['source','word_count','status']].describe(include='all').to_string())

valid = parse_df[parse_df['status'] == 'success']['word_count'].dropna()
print(f'\nWord count stats (successful parses only, n={len(valid)}):')
print(valid.describe().round(0).to_string())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(valid, bins=40, color='#4472C4', edgecolor='white', alpha=0.85)
ax.axvline(valid.median(), color='red', linestyle='--', label=f'Median: {valid.median():.0f}')
ax.axvline(valid.mean(),   color='orange', linestyle='--', label=f'Mean: {valid.mean():.0f}')
ax.set_xlabel('Word Count per Document')
ax.set_ylabel('Number of Documents')
ax.set_title('Word Count Distribution of Parsed Documents', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6 — Parse success / failure / empty breakdown
status_counts = parse_df['status'].value_counts()
print('Parse status breakdown:')
print(status_counts.to_string())

# Also flag zero-word-count as 'empty'
parse_df['effective_status'] = parse_df.apply(
    lambda r: 'empty' if r['status'] == 'success' and r.get('word_count', 1) == 0
              else r['status'], axis=1
)
eff_counts = parse_df['effective_status'].value_counts()
print('\nEffective status (empty = parsed but 0 words):')
print(eff_counts.to_string())

total = len(parse_df)
for st, cnt in eff_counts.items():
    print(f'  {st}: {cnt}/{total} ({cnt/total*100:.1f}%)')

In [ ]:
# Cell 7 — Year distribution of documents (extract year from 'date' column)
def extract_year(date_str):
    """Extract 4-digit year from various date formats, return NaN if missing."""
    if pd.isna(date_str):
        return np.nan
    m = re.search(r'\b(19|20)\d{2}\b', str(date_str))
    return int(m.group()) if m else np.nan

meta['year'] = meta['date'].apply(extract_year)
year_counts  = meta['year'].dropna().astype(int).value_counts().sort_index()
missing      = meta['year'].isna().sum()

print(f'Year range: {year_counts.index.min()} – {year_counts.index.max()}')
print(f'Missing year: {missing}/{len(meta)} documents')

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(year_counts.index, year_counts.values, color='#2ca02c', edgecolor='white', alpha=0.85)
ax.set_xlabel('Year')
ax.set_ylabel('Number of Documents')
ax.set_title('Year Distribution of Regulatory Documents', fontweight='bold')
ax.tick_params(axis='x', labelrotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 — QA item distribution: task_type and difficulty breakdown
with open(QA_JSON, encoding='utf-8') as f:
    qa_data = json.load(f)
qa_df = pd.DataFrame(qa_data)

print(f'QA dataset: {len(qa_df)} items')
print('\nTask type distribution:')
print(qa_df['task_type'].value_counts().to_string())
print('\nDifficulty distribution:')
print(qa_df['difficulty'].value_counts().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Task type
task_counts = qa_df['task_type'].value_counts()
task_labels = [t.replace('_', '\n') for t in task_counts.index]
axes[0].bar(task_labels, task_counts.values, color='#4472C4', edgecolor='white')
axes[0].set_title('QA Items by Task Type', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(task_counts.values):
    axes[0].text(i, v + 0.3, str(v), ha='center', fontweight='bold')

# Difficulty countplot using seaborn
diff_order = ['easy', 'medium', 'hard']
diff_palette = {'easy': '#4CAF50', 'medium': '#FF9800', 'hard': '#F44336'}
sns.countplot(data=qa_df, x='difficulty', order=diff_order,
              palette=diff_palette, ax=axes[1])
axes[1].set_title('QA Items by Difficulty', fontweight='bold')
axes[1].set_ylabel('Count')
for p in axes[1].patches:
    axes[1].text(p.get_x() + p.get_width()/2, p.get_height() + 0.3,
                 int(p.get_height()), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Summary statistics table (matches Table 8 in paper)
n_total  = len(qa_df)
n_sebi   = (qa_df['source'] == 'SEBI').sum()
n_rbi    = (qa_df['source'] == 'RBI').sum()

task_map = {
    'regulatory_interpretation': 'REG',
    'numerical_reasoning':       'NUM',
    'contradiction_detection':   'CON',
    'temporal_reasoning':        'TMP',
}
task_ns  = qa_df['task_type'].value_counts().rename(task_map)
diff_ns  = qa_df['difficulty'].value_counts()

n_docs_sebi = len(sebi_meta)
n_docs_rbi  = len(rbi_meta)

parse_success = (parse_df['status'] == 'success').sum()
parse_total   = len(parse_df)

summary = pd.DataFrame({
    'Statistic': [
        'Total QA pairs',
        '  From SEBI documents',
        '  From RBI documents',
        'REG (Regulatory Interpretation)',
        'NUM (Numerical Reasoning)',
        'CON (Contradiction Detection)',
        'TMP (Temporal Reasoning)',
        'Easy',
        'Medium',
        'Hard',
        'SEBI source documents',
        'RBI source documents',
        'Parse success rate',
    ],
    'Value': [
        n_total,
        n_sebi,
        n_rbi,
        task_ns.get('REG', 0),
        task_ns.get('NUM', 0),
        task_ns.get('CON', 0),
        task_ns.get('TMP', 0),
        diff_ns.get('easy', 0),
        diff_ns.get('medium', 0),
        diff_ns.get('hard', 0),
        n_docs_sebi,
        n_docs_rbi,
        f'{parse_success}/{parse_total} ({parse_success/parse_total*100:.1f}%)',
    ]
})

print('IndiaFinBench — Dataset Summary (Table 8)\n')
print(summary.to_string(index=False))